## **ANALISIS.SQL — Consultas Analíticas con CTEs y Window Functions**

Se implementan 5 consultas que responden las preguntas de negocio definidas por RetailNova.
Todas utilizan CTEs encadenadas y Window Functions con `PARTITION BY` y `ORDER BY`.

| # | Pregunta de negocio | Window Function usada |
|---|---|---|
| 1 | ¿Qué categorías generan más ingresos? | `RANK()`, `SUM() OVER()` |
| 2 | ¿Qué clientes generan más valor? | `DENSE_RANK()`, `NTILE(4)` |
| 3 | ¿Cómo evolucionan las ventas en el tiempo? | `SUM() OVER()`, `LAG()` |
| 4 | ¿Qué productos tienen baja rotación? | `PERCENT_RANK()`, `SUM() OVER()` |
| 5 | ¿Cómo se comportan los géneros por categoría? | `AVG() OVER(PARTITION BY)` |

In [1]:
%%sql
-- Propósito: Rankear las categorías de producto por total de ingresos y unidades vendidas.
-- Window Function: RANK() OVER (ORDER BY total_ingresos DESC)
-- CTE: agrega primero por categoría, luego aplica el ranking.
 
WITH ventas_por_categoria AS (
    SELECT
        product_category,
        COUNT(transaction_id)        AS total_transacciones,
        SUM(quantity)                AS total_unidades,
        SUM(total_amount)            AS total_ingresos,
        AVG(total_amount)            AS ticket_promedio
    FROM silver
    GROUP BY product_category
),
 
ranking_categorias AS (
    SELECT
        product_category,
        total_transacciones,
        total_unidades,
        ROUND(total_ingresos, 2)     AS total_ingresos,
        ROUND(ticket_promedio, 2)    AS ticket_promedio,
        RANK() OVER (
            ORDER BY total_ingresos DESC
        )                            AS ranking_ingresos,
        ROUND(
            total_ingresos * 100.0 / SUM(total_ingresos) OVER (), 2
        )                            AS pct_ingresos
    FROM ventas_por_categoria
)
 
SELECT
    ranking_ingresos,
    product_category,
    total_transacciones,
    total_unidades,
    total_ingresos,
    ticket_promedio,
    pct_ingresos
FROM ranking_categorias
ORDER BY ranking_ingresos;

StatementMeta(, c542e3ad-ab39-4d7e-ba77-207110d9f6ca, 2, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 7 fields>

## **Consulta 2 — ¿Qué clientes generan más valor?**
Identifica los clientes de mayor valor económico y los segmenta en 4 niveles 
de gasto para orientar estrategias de retención y fidelización.

**Lógica de la consulta:**
- `gasto_por_cliente` — agrega todas las transacciones por cliente calculando 
  compras totales, gasto acumulado, ticket promedio y compra máxima
- `clientes_rankeados` — aplica ranking y segmentación sobre los agregados

**Window Functions utilizadas:**
- `DENSE_RANK()` — ranking continuo sin saltos por empates en gasto total
- `NTILE(4)` — divide los clientes en 4 cuartiles de igual tamaño

**Segmentación resultante:**
| Cuartil | Segmento | Descripción |
|---|---|---|
| 1 | Premium | Top 25% de clientes por gasto |
| 2 | Alto | Entre 25% y 50% |
| 3 | Medio | Entre 50% y 75% |
| 4 | Bajo | Último 25% de menor gasto |

In [2]:
%%sql
-- Propósito: Identificar los clientes de mayor valor y segmentarlos por cuartil de gasto.
-- Window Function: DENSE_RANK() para ranking, NTILE(4) para segmentación en cuartiles.
-- CTE: agrega por cliente, luego clasifica.
 
WITH gasto_por_cliente AS (
    SELECT
        customer_id,
        gender,
        age,
        COUNT(transaction_id)     AS total_compras,
        SUM(total_amount)         AS gasto_total,
        AVG(total_amount)         AS gasto_promedio,
        MAX(total_amount)         AS compra_maxima
    FROM silver
    GROUP BY customer_id, gender, age
),
 
clientes_rankeados AS (
    SELECT
        customer_id,
        gender,
        age,
        total_compras,
        ROUND(gasto_total, 2)     AS gasto_total,
        ROUND(gasto_promedio, 2)  AS gasto_promedio,
        ROUND(compra_maxima, 2)   AS compra_maxima,
        DENSE_RANK() OVER (
            ORDER BY gasto_total DESC
        )                         AS ranking_cliente,
        -- Segmentación: 1 = top clientes, 4 = clientes de menor gasto
        NTILE(4) OVER (
            ORDER BY gasto_total DESC
        )                         AS cuartil_gasto
    FROM gasto_por_cliente
)
 
SELECT
    ranking_cliente,
    customer_id,
    gender,
    age,
    total_compras,
    gasto_total,
    gasto_promedio,
    compra_maxima,
    CASE cuartil_gasto
        WHEN 1 THEN 'Premium'
        WHEN 2 THEN 'Alto'
        WHEN 3 THEN 'Medio'
        WHEN 4 THEN 'Bajo'
    END                           AS segmento_cliente
FROM clientes_rankeados
ORDER BY ranking_cliente

StatementMeta(, c542e3ad-ab39-4d7e-ba77-207110d9f6ca, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 9 fields>

## **Consulta 3 — ¿Cómo evolucionan las ventas en el tiempo?**
Analiza la tendencia mensual de ventas calculando ingresos acumulados 
y variaciones respecto al mes anterior para detectar estacionalidad 
y tendencias de crecimiento.

**Lógica de la consulta:**
- `ventas_mensuales` — agrega transacciones, unidades e ingresos por mes
- `evolucion` — calcula métricas de comparación temporal sobre los agregados

**Window Functions utilizadas:**
- `SUM() OVER (ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)` — 
  acumulado progresivo de ingresos mes a mes
- `LAG()` — trae el valor del mes anterior para calcular la variación

**Métricas resultantes:**
| Columna | Descripción |
|---|---|
| `ingresos_acumulados` | Total acumulado desde el primer mes |
| `variacion_absoluta` | Diferencia en $ respecto al mes anterior |
| `variacion_pct` | Crecimiento porcentual mes a mes |

> 💡 `NULLIF(ingresos_mes_anterior, 0)` evita división por cero 
> en el primer mes de la serie.

In [3]:
%%sql
WITH ventas_mensuales AS (
    SELECT
        DATE_FORMAT(date, 'yyyy-MM')      AS anio_mes,
        COUNT(transaction_id)             AS total_transacciones,
        SUM(quantity)                     AS total_unidades,
        ROUND(SUM(total_amount), 2)       AS ingresos_mes
    FROM silver
    GROUP BY
        DATE_FORMAT(date, 'yyyy-MM')
),

evolucion AS (
    SELECT
        anio_mes,
        total_transacciones,
        total_unidades,
        ingresos_mes,
        ROUND(SUM(ingresos_mes) OVER (
            ORDER BY anio_mes
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ), 2)                             AS ingresos_acumulados,
        LAG(ingresos_mes) OVER (
            ORDER BY anio_mes
        )                                 AS ingresos_mes_anterior
    FROM ventas_mensuales
)

SELECT
    anio_mes,
    total_transacciones,
    total_unidades,
    ingresos_mes,
    ingresos_acumulados,
    ROUND(ingresos_mes - ingresos_mes_anterior, 2)                        AS variacion_absoluta,
    ROUND((ingresos_mes - ingresos_mes_anterior) * 100.0
          / NULLIF(ingresos_mes_anterior, 0), 2)                          AS variacion_pct
FROM evolucion
ORDER BY anio_mes;

StatementMeta(, c542e3ad-ab39-4d7e-ba77-207110d9f6ca, 4, Finished, Available, Finished, False)

<Spark SQL result set with 13 rows and 7 fields>

## **Consulta 4 — ¿Qué productos tienen baja rotación?**
Identifica las categorías con menor movimiento relativo de inventario 
por género, generando alertas accionables para decisiones de reabastecimiento 
y estrategia comercial.

**Lógica de la consulta:**
- `rotacion_por_categoria` — agrega unidades vendidas, transacciones e 
  ingresos por combinación de categoría y género
- `analisis_rotacion` — calcula la posición relativa de cada combinación 
  dentro de la distribución total de ventas

**Window Functions utilizadas:**
- `PERCENT_RANK()` — ubica cada categoría entre 0 y 1 dentro de la 
  distribución de unidades vendidas (0 = menor rotación, 1 = mayor)
- `SUM() OVER()` — calcula la participación porcentual de cada combinación 
  sobre el total de unidades

**Sistema de alertas:**
| Rango percentil | Alerta | Acción sugerida |
|---|---|---|
| < 0.25 | 🔴 Baja rotación | Revisar precio, promocionar o descontinuar |
| 0.25 – 0.75 | 🟡 Rotación media | Monitorear tendencia |
| > 0.75 | 🟢 Alta rotación | Garantizar reabastecimiento |


In [4]:
%%sql
-- Propósito: Identificar categorías con menor movimiento relativo para alertas de inventario.
-- Window Function: PERCENT_RANK() para ubicar cada categoría en la distribución de ventas.
--                  SUM() OVER() para calcular participación porcentual.
-- CTE: agrega por categoría y género, luego calcula posición relativa.
 
WITH rotacion_por_categoria AS (
    SELECT
        product_category,
        gender,
        SUM(quantity)                    AS unidades_vendidas,
        COUNT(transaction_id)            AS num_transacciones,
        ROUND(SUM(total_amount), 2)      AS ingresos_totales
    FROM silver
    GROUP BY product_category, gender
),
 
analisis_rotacion AS (
    SELECT
        product_category,
        gender,
        unidades_vendidas,
        num_transacciones,
        ingresos_totales,
        -- Posición percentil: 0 = menor rotación, 1 = mayor rotación
        ROUND(PERCENT_RANK() OVER (
            ORDER BY unidades_vendidas ASC
        ), 4)                            AS percentil_rotacion,
        -- Participación de cada combinación en el total de unidades
        ROUND(
            unidades_vendidas * 100.0 / SUM(unidades_vendidas) OVER (), 2
        )                                AS pct_unidades
    FROM rotacion_por_categoria
)
 
SELECT
    product_category,
    gender,
    unidades_vendidas,
    num_transacciones,
    ingresos_totales,
    pct_unidades,
    percentil_rotacion,
    -- Alerta de baja rotación: percentil menor a 0.25
    CASE
        WHEN percentil_rotacion < 0.25 THEN '🔴 Baja rotación'
        WHEN percentil_rotacion < 0.75 THEN '🟡 Rotación media'
        ELSE '🟢 Alta rotación'
    END                                  AS alerta_rotacion
FROM analisis_rotacion
ORDER BY percentil_rotacion ASC;

StatementMeta(, c542e3ad-ab39-4d7e-ba77-207110d9f6ca, 5, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 8 fields>

## **Consulta 5 — ¿Cómo se comportan los géneros por categoría?**
Compara el ticket promedio de cada combinación género-categoría contra 
tres referencias simultáneas para identificar segmentos sobre y bajo 
el promedio, orientando estrategias de pricing y marketing diferenciado.

**Lógica de la consulta:**
- `ticket_por_segmento` — agrega transacciones, unidades e ingresos 
  por cada combinación de género y categoría
- `comparacion_promedios` — calcula tres promedios de referencia en 
  paralelo usando `PARTITION BY` con distintos niveles de agregación

**Window Functions utilizadas:**
- `AVG() OVER (PARTITION BY gender)` — promedio de ticket dentro 
  de cada género, independiente de la categoría
- `AVG() OVER (PARTITION BY product_category)` — promedio de ticket 
  dentro de cada categoría, independiente del género
- `AVG() OVER ()` — promedio global sin ninguna partición

**Niveles de comparación:**
| Referencia | Descripción |
|---|---|
| `avg_ticket_por_genero` | ¿Este segmento gasta más o menos que su género? |
| `avg_ticket_por_categoria` | ¿Este segmento gasta más o menos que su categoría? |
| `avg_ticket_global` | ¿Este segmento está sobre o bajo el promedio general? |


In [5]:
%%sql
-- Propósito: Comparar el comportamiento de compra entre géneros por categoría
--            e identificar desviaciones respecto al promedio general.
-- Window Function: AVG() OVER (PARTITION BY) para promedios por grupo y global.
-- CTE: agrega por género y categoría, luego compara contra promedios de referencia.
 
WITH ticket_por_segmento AS (
    SELECT
        gender,
        product_category,
        COUNT(transaction_id)            AS total_transacciones,
        SUM(quantity)                    AS total_unidades,
        ROUND(SUM(total_amount), 2)      AS ingresos_totales,
        ROUND(AVG(total_amount), 2)      AS ticket_promedio
    FROM silver
    GROUP BY gender, product_category
),
 
comparacion_promedios AS (
    SELECT
        gender,
        product_category,
        total_transacciones,
        total_unidades,
        ingresos_totales,
        ticket_promedio,
        -- Promedio de ticket por género (independiente de categoría)
        ROUND(AVG(ticket_promedio) OVER (
            PARTITION BY gender
        ), 2)                            AS avg_ticket_por_genero,
        -- Promedio de ticket por categoría (independiente de género)
        ROUND(AVG(ticket_promedio) OVER (
            PARTITION BY product_category
        ), 2)                            AS avg_ticket_por_categoria,
        -- Promedio global de ticket
        ROUND(AVG(ticket_promedio) OVER (), 2) AS avg_ticket_global
    FROM ticket_por_segmento
)
 
SELECT
    gender,
    product_category,
    total_transacciones,
    total_unidades,
    ingresos_totales,
    ticket_promedio,
    avg_ticket_por_genero,
    avg_ticket_por_categoria,
    avg_ticket_global,
    -- Desviación del ticket del segmento vs el promedio global
    ROUND(ticket_promedio - avg_ticket_global, 2)  AS desviacion_vs_global,
    CASE
        WHEN ticket_promedio > avg_ticket_global THEN '⬆ Sobre promedio'
        WHEN ticket_promedio < avg_ticket_global THEN '⬇ Bajo promedio'
        ELSE '➡ En promedio'
    END                                            AS posicion_vs_global
FROM comparacion_promedios
ORDER BY gender, ingresos_totales DESC;

StatementMeta(, c542e3ad-ab39-4d7e-ba77-207110d9f6ca, 6, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 11 fields>